In [12]:
print("'Soham',son of pranav ")

'Soham',son of pranav 


In [13]:
class Channel:
    def __init__(self):
        self.valid = False
        self.ready = False
        self.data = None

    def send(self, data):
        self.valid = True
        self.data = data

    def handshake(self):
        return self.valid and self.ready

    def clear(self):
        self.valid = False
        self.data = None


class AXIInterface:
    def __init__(self):
        # Read channels
        self.ar = Channel()
        self.r  = Channel()

        # Write channels
        self.aw = Channel()
        self.w  = Channel()
        self.b  = Channel()

class Memory:
    def __init__(self, size):
        self.mem = [0] * size

    def read(self, addr, size):
        return self.mem[addr:addr+size]

    def write(self, addr, data):
        for i, d in enumerate(data):
            self.mem[addr + i] = d


class AXISlave:
    def __init__(self, axi, memory):
        self.axi = axi
        self.memory = memory

    def tick(self):
        # READ ADDRESS
        if self.axi.ar.handshake():
            req = self.axi.ar.data
            addr = req["addr"]
            length = req["len"]

            data = self.memory.read(addr, length)

            # send all data (simplified)
            self.axi.r.send({
                "data": data,
                "resp": 0,
                "last": True
            })

        # WRITE DATA
        if self.axi.w.handshake():
            write = self.axi.w.data
            self.memory.write(write["addr"], write["data"])

            self.axi.b.send({"resp": 0})


class AXIMaster:
    def __init__(self, axi):
        self.axi = axi

    def read(self, addr, length):
        self.axi.ar.send({"addr": addr, "len": length})

    def get_read_data(self):
        if self.axi.r.handshake():
            data = self.axi.r.data
            self.axi.r.clear()
            return data

    def write(self, addr, data):
        self.axi.aw.send({"addr": addr})
        self.axi.w.send({"addr": addr, "data": data})

    def get_write_resp(self):
        if self.axi.b.handshake():
            resp = self.axi.b.data
            self.axi.b.clear()
            return resp
        

axi = AXIInterface()
mem = Memory(1024)
slave = AXISlave(axi, mem)
master = AXIMaster(axi)

# Connect ready signals (simplified)
axi.ar.ready = True
axi.w.ready  = True
axi.r.ready  = True
axi.b.ready  = True

# Example transaction
master.read(10, 4)

for cycle in range(5):
    slave.tick()

    data = master.get_read_data()
    if data:
        print("Received:", data)



Received: {'data': [0, 0, 0, 0], 'resp': 0, 'last': True}
Received: {'data': [0, 0, 0, 0], 'resp': 0, 'last': True}
Received: {'data': [0, 0, 0, 0], 'resp': 0, 'last': True}
Received: {'data': [0, 0, 0, 0], 'resp': 0, 'last': True}
Received: {'data': [0, 0, 0, 0], 'resp': 0, 'last': True}
